# 03 — Certainty Propagation & Evidence Tree

Demonstrates the full certainty pipeline using only public SDK + Audit API:
- Rules with `condition_weights`
- Facts with `confidence`
- Automatic certainty routing at evaluation time
- Evidence tree with confidence carriers (via `AuditQuery`)
- Certainty summary: bottleneck vs. additive aggregation
- Narrative with ranked conditions

**Prerequisites:** [01](01_sdk_basics.ipynb) and [02](02_rules_and_derivations.ipynb).  
**Next:** [04_ecss_souffle_compliance.ipynb](04_ecss_souffle_compliance.ipynb)

## 0. Imports

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve().parent / "src"))

In [ ]:
from __future__ import annotations
import tempfile
from pathlib import Path
from pprint import pprint

from factpy_kernel.sdk import (
    SDKStore, SDKRegistry, Entity, Identity, Field,
    Rule, Derivation, Pred, vars as sdk_vars,
)
from factpy_kernel.sdk.dsl.rule import RuleRef
from factpy_kernel.adapters.souffle.package import ExportOptions
from factpy_kernel.audit import AuditQuery, load_audit_package

## 1. Schema & Rule with `condition_weights`

`condition_weights` maps body clause positions to importance weights.
They drive the certainty propagation pipeline.

In [ ]:
class User(Entity):
    user_id: str = Identity(primary_key=True)
    locale: str = Identity()
    name: str = Field(cardinality="single")
    tag: str = Field(cardinality="multi")
    score: str = Field(cardinality="single")

sdk = SDKStore([User])

with sdk_vars("u", "tag", "score") as (u, tag, score):
    child_rule = Rule(
        id="q.qualified_user", version="1.0.0",
        select=[u, tag],
        where=[Pred("user:tag", u, tag), Pred("user:score", u, score)],
        expose=True,
        condition_weights={"b0.a0": 0.9, "b0.a1": 0.4},
    )

print("Rule:", child_rule.id)
print("Condition weights:", child_rule.condition_weights)

## 2. Registry & Seed Data with `confidence`

Fact-level confidence values feed into certainty propagation.

In [ ]:
registry_dir = tempfile.mkdtemp(prefix="certainty_demo_")
registry = SDKRegistry(root_dir=Path(registry_dir))
registry.apply_schema_classes([User])
registry.register_rule(child_rule, schema_ir=sdk.schema_ir)

with sdk.batch() as tx:
    u1 = tx.entity(User, user_id="u-001", locale="en")
    u1.name.set("Alice")
    u1.tag.add("vip", meta={"confidence": 0.95})
    u1.score.set("85", meta={"confidence": 0.6})
    tx.commit()

u1_ref = sdk.ref(User, user_id="u-001", locale="en")
print("Facts: tag='vip' (conf=0.95), score='85' (conf=0.6)")

## 3. Evaluate with Auto Certainty Routing

When a rule has `condition_weights` and facts have `confidence`, the framework
automatically routes to certainty-aware evaluation at candidate creation time.

In [ ]:
with sdk_vars("u", "tag") as (u, tag):
    derivation = Derivation(
        id="drv.certainty_demo", version="1.0.0",
        where=[RuleRef(child_rule)(u, tag), tag == "vip"],
        target="user:tag",
        head_vars=[u, tag],
    )

cands = sdk.evaluate(derivation, mode="native", registry=registry)
candidate = cands[0]
candidate_id = candidate.candidate_id
print(f"confidence: {candidate.confidence}")
print(f"confidence_kind: {candidate.confidence_kind} <- auto-routed")

## 4. Accept + Export Audit Package

Accept the candidate, then export the audit package.
The audit package contains evidence trees and certainty summaries.

In [ ]:
sdk.accept(candidate, approved_by="demo", note="certainty demo")

audit_dir = tempfile.mkdtemp(prefix="certainty_audit_")
sdk.export_package(audit_dir, ExportOptions(package_kind="audit"), registry=registry)

pkg = load_audit_package(audit_dir)
aq = AuditQuery(pkg)
print(f"Audit package: {len(aq.list_candidates())} candidates")

## 5. Evidence Tree: Confidence Carriers

Each witness assertion carries its `confidence` and `condition_confidence`.

In [ ]:
tree = aq.get_candidate_evidence_tree(candidate_id)

def print_tree(node, indent=0):
    prefix = "  " * indent
    kind = node.get("node_kind", "?")
    label = kind
    if kind == "predicate_witness_group":
        label += f"  pred_id={node.get('pred_id', '?')}"
        cc = node.get('condition_confidence')
        if cc is not None: label += f"  condition_confidence={cc}"
    elif kind == "assertion_fact":
        claims = node.get("claim_args", [])
        label += f"  [{', '.join(c.get('val','?') for c in claims)}]"
        conf = node.get("confidence")
        if conf is not None: label += f"  confidence={conf}"
    elif kind == "rule_ref":
        label += f"  {node.get('rule_ref_id', '?')} v{node.get('rule_ref_version', '?')}"
    print(f"{prefix}- {label}")
    for child in node.get("children", []):
        print_tree(child, indent + 1)

if tree and "root" in tree:
    print("=== Evidence Tree ===")
    print_tree(tree["root"])
else:
    print("No evidence tree (candidate may not have certainty routing)")

## 6. Certainty Summary: Bottleneck (Default)

Bottleneck aggregation: `aggregate = min(condition impacts)`.

In [ ]:
cs_bottleneck = aq.get_candidate_certainty_summary(candidate_id)

if cs_bottleneck:
    print("=== Certainty Summary (bottleneck) ===")
    pprint(cs_bottleneck)
else:
    print("No certainty summary (rule may lack condition_weights)")

## 7. Evidence Tree Narrative

In [ ]:
narrative = aq.get_candidate_evidence_tree_narrative(candidate_id)

if narrative:
    print("=== Narrative ===")
    for key in ["headline", "overview_lines", "evidence_lines", "certainty_lines"]:
        val = narrative.get(key)
        if val:
            if isinstance(val, list):
                for line in val: print(f"  {line}")
            else:
                print(f"  {val}")
    bottleneck = narrative.get("certainty_bottleneck")
    if bottleneck:
        print(f"  bottleneck: {bottleneck}")
else:
    print("No narrative available")

## 8. Negative Case: No `condition_weights`

Without `condition_weights`, the certainty pipeline is not activated.

In [ ]:
with sdk_vars("u", "tag", "score") as (u, tag, score):
    plain_rule = Rule(id="q.plain_rule", version="1.0.0", select=[u, tag],
        where=[Pred("user:tag", u, tag), Pred("user:score", u, score)], expose=True)
registry.register_rule(plain_rule, schema_ir=sdk.schema_ir)

with sdk_vars("u", "tag") as (u, tag):
    plain_drv = Derivation(id="drv.plain", version="1.0.0",
        where=[RuleRef(plain_rule)(u, tag), tag == "vip"],
        target="user:tag", head_vars=[u, tag])

plain_cands = sdk.evaluate(plain_drv, mode="native", registry=registry)
if plain_cands:
    pc = plain_cands[0]
    print(f"confidence_kind: {pc.confidence_kind}")
    print(">>> No condition_weights -> no certainty routing -> no certainty delivery")

---
**Next:** [04_ecss_souffle_compliance.ipynb](04_ecss_souffle_compliance.ipynb)